In [1]:
import cv2
import numpy as np

## 1. Detecting Features

In [2]:
cap = cv2.VideoCapture(0)

imgTarget = cv2.imread('Images/Quad.jpg')
myVid = cv2.VideoCapture('Videos/Nights.mp4')

s1, imgVideo = myVid.read()
h, w, c = imgTarget.shape
imgVideo = cv2.resize(imgVideo, (w,h))

orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(imgTarget, None)

while True:
    s2, frame = cap.read()
    kp2, des2 = orb.detectAndCompute(frame, None)
    frame = cv2.drawKeypoints(frame, kp2, None)
    
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1,des2, k=2)
    
    good = []
    for m,n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)
    
    imgFeatures = cv2.drawMatches(imgTarget, kp1, frame, kp2, good, None, flags=2)
    
    cv2.imshow('imgTarget', imgTarget)
    cv2.imshow('myVid', imgVideo)
    cv2.imshow('Frame', frame)
    cv2.imshow('imgFeatures', imgFeatures)
    
    
    if cv2.waitKey(1) & 0xFF == 27:
        break
    
cap.release()
cv2.destroyAllWindows()

In [3]:
cap.release()
cv2.destroyAllWindows()

## 2. Bounding Box

In [4]:
cap = cv2.VideoCapture(0)

imgTarget = cv2.imread('Images/Quad.jpg')
myVid = cv2.VideoCapture('Videos/Nights.mp4')

s1, imgVideo = myVid.read()
h, w, c = imgTarget.shape
imgVideo = cv2.resize(imgVideo, (w,h))

orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(imgTarget, None)

while True:
    s2, frame = cap.read()
    kp2, des2 = orb.detectAndCompute(frame, None)
    frame = cv2.drawKeypoints(frame, kp2, None)
    
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1,des2, k=2)
    
    good = []
    for m,n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)
    # print(len(good))
    
    imgFeatures = cv2.drawMatches(imgTarget, kp1, frame, kp2, good, None, flags=2)
    
    if len(good) > 90:
        srcPts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dstPts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        
        matrix, mask = cv2.findHomography(srcPts, dstPts, cv2.RANSAC, 5)
        
        # print(matrix)
        
        pts = np.float32([[0,0], [0,h], [w,h], [w,0]]).reshape(-1,1,2)
        
        dst = cv2.perspectiveTransform(pts,matrix)
        
        frame2 = cv2.polylines(frame, [np.int32(dst)], True, (255,0,255), 3)
        
        cv2.imshow('Frame2', frame2)
        
        imgWarp = cv2.warpPerspective(imgVideo, matrix, (frame.shape[1],frame.shape[0]))
    
        cv2.imshow('imgWarp', imgWarp)
        
    cv2.imshow('imgTarget', imgTarget)
    cv2.imshow('myVid', imgVideo)
    cv2.imshow('Frame', frame)
    cv2.imshow('imgFeatures', imgFeatures) 
    
    if cv2.waitKey(1) & 0xFF == 27:
        break
    
cap.release()
cv2.destroyAllWindows()

In [5]:
cap.release()
cv2.destroyAllWindows()

## 3. Overlay of Video

In [6]:
cap = cv2.VideoCapture(0)

imgTarget = cv2.imread('Images/Quad.jpg')
myVid = cv2.VideoCapture('Videos/Fly.mp4')

detection = False
frameCounter = 0

s1, imgVideo = myVid.read()
h, w, c = imgTarget.shape
imgVideo = cv2.resize(imgVideo, (w,h))

orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(imgTarget, None)

while True:
    s2, frame = cap.read()
    imgAug = frame.copy()
    kp2, des2 = orb.detectAndCompute(frame, None)
    # frame = cv2.drawKeypoints(frame, kp2, None)
    
    if detection == False:
        myVid.set(cv2.CAP_PROP_POS_FRAMES, 0)
        frameCounter = 0
    else:
        if frameCounter == myVid.get(cv2.CAP_PROP_FRAME_COUNT):
            myVid.set(cv2.CAP_PROP_POS_FRAMES, 0)
            frameCounter = 0
        s3, imgVideo = myVid.read()
        imgVideo = cv2.resize(imgVideo, (w,h))
        
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1,des2, k=2)
    
    good = []
    for m,n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)
    # print(len(good))
    
    imgFeatures = cv2.drawMatches(imgTarget, kp1, frame, kp2, good, None, flags=2)
    
    if len(good) > 90:
        detection =  True
        srcPts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dstPts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        
        matrix, mask = cv2.findHomography(srcPts, dstPts, cv2.RANSAC, 5)
        
        # print(matrix)
        
        pts = np.float32([[0,0], [0,h], [w,h], [w,0]]).reshape(-1,1,2)
        
        dst = cv2.perspectiveTransform(pts,matrix)
        
        frame2 = cv2.polylines(frame, [np.int32(dst)], True, (255,0,255), 3)
        # cv2.imshow('Frame2', frame2)
        
        imgWarp = cv2.warpPerspective(imgVideo, matrix, (frame.shape[1],frame.shape[0]))
        # cv2.imshow('imgWarp', imgWarp)
        
        maskNew = np.zeros((frame.shape[0], frame.shape[1]), np.uint8)
        cv2.fillPoly(maskNew, [np.int32(dst)], (255,255,255))
        maskInv = cv2.bitwise_not(maskNew)
        
        imgAug = cv2.bitwise_and(imgAug, imgAug, mask=maskInv)
        
        imgAug = cv2.bitwise_or(imgWarp, imgAug)
    else: 
        detection = False
        
    # cv2.imshow('imgTarget', imgTarget)
    # cv2.imshow('myVid', imgVideo)
    # cv2.imshow('Frame', frame)
    # cv2.imshow('imgFeatures', imgFeatures) 
    
    cv2.imshow('imgAug', imgAug)
    
    if cv2.waitKey(1) & 0xFF == 27:
        break
        
    frameCounter += 1
    
cap.release()
cv2.destroyAllWindows()

In [7]:
cap.release()
cv2.destroyAllWindows()